# Matrícula por estudiante 2024: descompresión, conversión a parquet y conteo por RBD

Pipeline reproducible: descomprime `datos/Matricula-por-estudiante-2024.rar`, convierte el CSV a parquet, y calcula el conteo de matrículas por establecimiento (RBD).

In [1]:
import subprocess
from pathlib import Path

import pandas as pd
import pyarrow.csv as pcsv
import pyarrow.parquet as pq

ruta_datos = Path("datos")
rar_path = ruta_datos / "Matricula-por-estudiante-2024.rar"
csv_path = ruta_datos / "20240913_Matrícula_unica_2024_20240430_WEB.CSV"
parquet_path = ruta_datos / "matricula_estudiante_2024.parquet"


## 1. Descomprimir el .rar

Requiere `unar` instalado (`brew install unar`). Solo se extrae si el CSV todavía no existe en `datos/`.

In [2]:
if not csv_path.exists():
    subprocess.run(
        ["unar", "-o", str(ruta_datos), str(rar_path)],
        check=True,
    )
    print("Descomprimido en", ruta_datos)
else:
    print("El CSV ya existe, se omite la descompresión:", csv_path)


El CSV ya existe, se omite la descompresión: datos/20240913_Matrícula_unica_2024_20240430_WEB.CSV


## 2. Transformar el CSV a parquet

In [3]:
tabla = pcsv.read_csv(csv_path, parse_options=pcsv.ParseOptions(delimiter=";"))
pq.write_table(tabla, parquet_path, compression="snappy")

print(f"{tabla.num_rows} filas, {tabla.num_columns} columnas -> {parquet_path}")


3582943 filas, 37 columnas -> datos/matricula_estudiante_2024.parquet


## 3. Conteo agrupado de matrículas por RBD

In [4]:
matricula = pd.read_parquet(parquet_path, columns=["RBD", "NOM_RBD", "MRUN"])

matricula_por_rbd = (
    matricula.groupby(["RBD", "NOM_RBD"])["MRUN"]
    .count()
    .reset_index(name="n_matriculas")
    .sort_values("n_matriculas", ascending=False)
)

matricula_por_rbd.head(10)


,RBD,NOM_RBD,n_matriculas
39,52,LICEO BICENTENARIO DOMINGO SANTA MARIA,3962
9523,25369,COLEGIO PARTICULAR ALICANTE,3663
5879,10487,COLEGIO POLIVALENTE DOMINGO MATTE MESIAS,3648
3407,5654,INSTITUTO CLARET,3227
5577,9959,COMPLEJO EDUCACIONAL MAIPU,3208
9193,24685,COLEGIO PIAMARTA,3197
4872,8485,LICEO INSTITUTO NACIONAL,3155
9772,25749,COLEGIO PART. ALICANTE DEL ROSAL,3133
10483,31327,COLEGIO ALICANTE DEL VALLE,3058
9288,24946,COLEGIO JORGE HUNEEUS ZEGERS,3051


In [5]:
suma_n_matriculas = matricula_por_rbd["n_matriculas"].sum()
assert suma_n_matriculas == tabla.num_rows, (
    f"La suma de n_matriculas ({suma_n_matriculas}) no coincide con las filas de tabla ({tabla.num_rows})"
)
print(f"OK: suma de n_matriculas ({suma_n_matriculas}) == filas de tabla ({tabla.num_rows})")


OK: suma de n_matriculas (3582943) == filas de tabla (3582943)


In [6]:
matricula_por_rbd_path = ruta_datos / "matricula_por_rbd_2024.parquet"
matricula_por_rbd.to_parquet(matricula_por_rbd_path, index=False)

print(f"{len(matricula_por_rbd)} RBD -> {matricula_por_rbd_path}")


11049 RBD -> datos/matricula_por_rbd_2024.parquet
